In [1]:
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect("../data/checking-logs.sqlite")
schema = pd.read_sql("PRAGMA table_info(test)", conn)
print(schema)

   cid             name       type  notnull dflt_value  pk
0    0              uid       TEXT        0       None   0
1    1          labname       TEXT        0       None   0
2    2  first_commit_ts  TIMESTAMP        0       None   0
3    3    first_view_ts  TIMESTAMP        0       None   0


In [ ]:
query = '''
    SELECT time, AVG(diff) AS avg_diff
    FROM (
        SELECT uid,
               CAST((julianday(t.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24 AS INTEGER) AS diff,
               CASE WHEN t.first_commit_ts < t.first_view_ts THEN 'before' ELSE 'after' END AS time
        FROM test t
        LEFT JOIN deadlines d ON t.labname = d.labs
        WHERE NOT t.labname = 'project1'
    )
    WHERE uid IN (
        SELECT uid
        FROM (
            SELECT uid,
                   CASE WHEN t.first_commit_ts < t.first_view_ts THEN 'before' ELSE 'after' END AS time
            FROM test t
            LEFT JOIN deadlines d ON t.labname = d.labs
            WHERE NOT t.labname = 'project1'
        )
        GROUP BY uid
        HAVING COUNT(DISTINCT time) = 2
    )
    GROUP BY time
'''

test_results = pd.read_sql(query, conn)
test_results

,time,avg_diff
0,after,-104.6000
1,before,-60.5625


In [4]:
query = '''
    SELECT time, AVG(diff) AS avg_diff
    FROM (
        SELECT uid,
               CAST((julianday(c.first_commit_ts) - julianday(datetime(d.deadlines, 'unixepoch'))) * 24 AS INTEGER) AS diff,
               CASE WHEN c.first_commit_ts < c.first_view_ts THEN 'before' ELSE 'after' END AS time
        FROM control c
        LEFT JOIN deadlines d ON c.labname = d.labs
        WHERE NOT c.labname = 'project1'
    )
    WHERE uid IN (
        SELECT uid
        FROM (
            SELECT uid,
                   CASE WHEN c.first_commit_ts < c.first_view_ts THEN 'before' ELSE 'after' END AS time
            FROM control c
            LEFT JOIN deadlines d ON c.labname = d.labs
            WHERE NOT c.labname = 'project1'
        )
        GROUP BY uid
        HAVING COUNT(DISTINCT time) = 2
    )
    GROUP BY time
'''

control_results = pd.read_sql(query, conn)
control_results

,time,avg_diff
0,after,-117.636364
1,before,-99.464286


In [5]:
conn.close()